# Faza testowa - wyniki eksperymentów

Czyta gotowe katalogi przebiegów z `results/runs/` i pliki adnotacji, i pokazuje, jak zachowuje się każdy komponent na zbiorze testowym. Nie liczy nic na nagraniach i nie ładuje żadnego modelu, więc wykonuje się w sekundy i można go uruchamiać po każdym kolejnym przebiegu. Bloki bez przebiegu nie wywalają się, tylko wypisują, czego brakuje.

Zbiorcze zestawienia, wrażliwość na wagi i kontrola kompletności są w `test_summary.ipynb`, wszystkie miary wszystkich konfiguracji w `test_appendix.ipynb`.

**Wymaga:** przebiegów części testowej, kolejno (szczegóły w `docs/04_instrukcja_dev.md` i `docs/05_instrukcja_test.md`):

```powershell
python scripts/make_configs.py --execute
python scripts/run_features.py  configs/full_office.yaml --split test    # oraz tbbt, vatex
python scripts/run_experiment.py configs/e2a_office.yaml --split test    # BAZA, trzy zbiory
python scripts/run_experiment.py configs/e2c_office.yaml --split test    # E2-C, E3-B, ..., E6-B
python scripts/run_experiment.py configs/full_office.yaml --split test   # potok pelny
python scripts/run_experiment.py configs/full_no_objects_office.yaml --split test
```

Rodziny `full_no_*` i `full_swap_*` dają wkład krańcowy i efekt podmiany; bez nich odpowiednie kolumny zostają puste, a reszta tabel liczy się normalnie.

Wartości Recall@K są w procentach, różnice w punktach procentowych, separatorem dziesiętnym jest przecinek. Nawias kwadratowy to 95% przedział ufności różnicy: dla seriali liczony po odcinkach-klastrach, dla VATEX-a bootstrapem po klipach. Liczba bez przedziału (`n=...`) oznacza, że jednostek wnioskowania było za mało, żeby go policzyć. Kolumna "polaczone" dotyczy wyłącznie seriali - klipy VATEX-a nie wchodzą do puli odcinków.

**Zapisuje:** nic, tylko wypisuje.

**Dalej:** `test_summary.ipynb` - zestawienia przekrojowe, wrażliwość na wagi i kontrola kompletności adnotacji.

In [ ]:
import importlib
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.annotation import tags
from src.data import queries as queries_module
from src.evaluation import compare, metrics, tables
from src.evaluation import runs as runs_module
from src.utils import experiments as exp

for module in (tags, queries_module, compare, metrics, runs_module, tables, exp):
    importlib.reload(module)      # the kernel keeps a once-imported module in memory
load_queries = queries_module.load_queries

SPLIT = "test"
METRIC = "recall@10"

# Which label means what, which pair a contribution is a difference of, which
# datasets a signal exists on: all of it lives in src/utils/experiments.py, so a
# table here and a generated configuration cannot disagree.
DATASETS = list(exp.DATASETS)
LABEL = exp.DATASET_NAMES
BASE, FULL = exp.BASE, exp.FULL


def name(label):
    """Display name of a label; the baseline reads BAZA outside E2."""
    return exp.display(label, base_as_name=False)


# one loader for the whole repository (src/evaluation/runs.py): the newest
# directory of a (label, dataset) wins, and nothing here reimplements that
runs = runs_module.load_runs(SPLIT)


def available(labels):
    """Runs of the given labels, on the datasets ALL of them cover."""
    present, shared, skipped = runs_module.available(runs, labels)
    for label, reason in skipped:
        print(f"  {label}: {reason}")
    return runs_module.select(runs, present, shared), shared


def missing(labels):
    """Prints what has to be run for a block to have anything to show."""
    gaps = [label for label in labels if label not in runs]
    if gaps:
        print(f"brak przebiegow: {', '.join(gaps)} - patrz naglowek notatnika")
    return bool(gaps)


def subsets(spec, datasets):
    """``(matching, rest, without)`` per dataset for one subset specification.

    ``without`` lists the datasets whose matching side is empty, which for a
    requirement tag means the tag is not assigned there at all -- VATEX carries
    no `wymaga_osoby` and no `wymaga_mimiki` (docs/03). An empty side is an
    answer, so the caller reports it instead of the block failing.
    """
    matching, rest, without = {}, {}, []
    for dataset in datasets:
        try:
            matching[dataset], rest[dataset] = compare.subset_ids(dataset, SPLIT, spec)
        except (FileNotFoundError, ValueError) as problem:
            without.append(dataset)
            print(f"  {LABEL[dataset]}: {problem}")
            continue
        if not matching[dataset]:
            without.append(dataset)
            matching.pop(dataset)
            rest.pop(dataset)
    return matching, rest, without


def on(runs_of, datasets):
    """The runs of one label narrowed to the given datasets.

    A contrast must not reach a dataset whose subset is empty: there
    ``matching`` and ``rest`` would both be None, both sides would be the whole
    query set and the difference of differences would come out as a clean zero
    that means nothing.
    """
    return {dataset: runs_of[dataset] for dataset in datasets if dataset in runs_of}


def variant_rows(group, columns, reference_label, labels):
    """The standard shape of an experiment table: value, delta, pooled delta."""
    reference = group[reference_label]
    results = {label: compare.compare_variant(group[label], reference, METRIC)
               for label in labels if label in group and label != reference_label}
    header = ["Konfiguracja"]
    for dataset in columns:
        header += [f"{LABEL[dataset]} R@10", f"{LABEL[dataset]} delta [95% PU]"]
    header += ["Polaczone delta [95% PU]"]

    rows = []
    for label in labels:
        if label not in group:
            continue
        cells = [name(label)]
        for dataset in columns:
            if label == reference_label:
                cells += [tables.percent(compare.query_mean(
                    reference[dataset]["per_query"], METRIC)), "-"]
            else:
                entry = results[label]["per_dataset"][dataset]
                cells += [tables.percent(entry["value"]), tables.interval(entry)]
        cells += ["-" if label == reference_label
                  else tables.interval(results[label]["pooled"])]
        rows.append(cells)
    return header, rows, results


def contribution_rows(signal):
    """Marginal contribution and swap effect of one signal, per dataset.

    Both are ordinary comparisons of two configurations -- the full pipeline
    against the full pipeline with that one signal removed, and against the full
    pipeline with its mechanism swapped -- so the pairs come from
    experiments.py and the arithmetic from compare.py. Neither is computed here.
    """
    where = list(exp.datasets_of(signal))
    rows = []
    kinds = [("wklad krancowy", exp.marginal_pair(signal))]
    if signal in exp.SWAPPED_SIGNALS:
        kinds.append(("efekt podmiany", exp.swap_pair(signal)))
    for title, (candidate, reference) in kinds:
        group, _ = available([candidate, reference])
        if len(group) < 2:
            rows.append([title, *["-"] * (len(where) + 1)])
            continue
        result = compare.compare_variant(group[candidate], group[reference], METRIC)
        rows.append([f"{title} ({name(reference)})"]
                    + [tables.points(result["per_dataset"][d]["mean"])
                       if d in result["per_dataset"] else "-" for d in where]
                    + [tables.interval(result["pooled"])])
    return where, rows


def contribution_table(signal, caption):
    """Prints the contribution rows of one signal under its own heading.

    The columns are the datasets the signal EXISTS on. A dataset it is not
    measured on gets no column at all: a column of dashes reads as "measured and
    came out empty", which is the one thing it does not mean.
    """
    where, rows = contribution_rows(signal)
    outside = [d for d in DATASETS if d not in where]
    tables.show(caption, ["Wielkosc"] + [LABEL[d] for d in where]
                + ["Polaczone [95% PU]"], rows,
                note="Wklad krancowy: potok pelny wobec potoku bez tego sygnalu. "
                     "Efekt podmiany: potok pelny wobec potoku z drugim mechanizmem "
                     "tego sygnalu. Obie wielkosci to zwykle roznice dwoch "
                     "konfiguracji, liczone tak samo jak reszta tabel."
                     + (f" Poza pomiarem tego sygnalu: "
                        f"{', '.join(LABEL[d] for d in outside)} - dlatego nie ma tu "
                        "tej kolumny." if outside else ""))


def full_block(signal, columns):
    """The full-pipeline block of an experiment table, or no rows at all.

    The marginal contribution and the swap effect as ordinary rows, each
    measured against the full pipeline itself. One rule then reads the whole
    table -- a row minus the reference of its block -- which puts both with the
    sign opposite to the summary table, where they are FULL minus the row.
    """
    in_full = [FULL, exp.no_label(signal)]
    if signal in exp.SWAPPED_SIGNALS:
        in_full.append(exp.swap_label(signal))
    group, shared = available(in_full)
    if len(group) < len(in_full) or list(shared) != list(columns):
        print(f"blok potoku PELNEGO pominiety: brak kompletu przebiegow "
              f"{', '.join(in_full)} na zbiorach {', '.join(columns)}")
        return []
    _, rows, _ = variant_rows(group, columns, FULL, in_full)
    return ["potok PELNY - odniesieniem bloku jest PELNY"] + rows


print(f"przebiegi znalezione dla czesci {SPLIT!r}:")
for label in sorted(runs):
    print(f"  {label:<24}{', '.join(sorted(runs[label]))}")
if not runs:
    print("  (jeszcze zadnego)")


## 1. Charakterystyka zbioru testowego

Wiersze znaczników i złożoności powstają z plików zapytań, więc ten blok działa jeszcze zanim ruszy pierwszy przebieg. Wiersze "fragmentów w kolekcji" i "potok pełny Recall@10" pochodzą z metadanych przebiegów i wypełniają się dopiero razem z nimi.

Znacznika, którego zbiór nie przypisuje, nie ma w tabeli jako zera - jest jako `-` (`docs/02_dane_i_znaczniki.md`).

In [ ]:
from src.utils.vocabulary import COMPLEXITY_VALUES, REQUIREMENT_TAGS

statistics, sizes, recall = {}, {}, {}
for dataset in DATASETS:
    try:
        collection = load_queries(dataset, SPLIT)
    except FileNotFoundError as problem:
        print(f"{LABEL[dataset]}: {problem}")
        continue
    statistics[dataset] = tags.statistics(collection, tags.columns_for(dataset))
    base_run = runs.get(BASE, {}).get(dataset)
    sizes[dataset] = base_run["metrics"].get("collection_size") if base_run else None
    full_run = runs.get(FULL, {}).get(dataset)
    recall[dataset] = (compare.query_mean(full_run["per_query"], METRIC)
                       if full_run else None)

if statistics:
    rows = [["Zapytan"] + [statistics[d]["count"] if d in statistics else "-"
                           for d in DATASETS]]
    for tag in REQUIREMENT_TAGS:
        rows.append([tag] + [statistics[d]["requirements"].get(tag, "-")
                             if d in statistics else "-" for d in DATASETS])
    for value in COMPLEXITY_VALUES:
        rows.append([f"zlozonosc {value}"]
                    + [statistics[d]["complexity"].get(value, "-")
                       if d in statistics else "-" for d in DATASETS])
    rows.append(["Fragmentow w kolekcji"]
                + [tables.number(sizes[d], 0) if sizes.get(d) is not None else "-"
                   for d in DATASETS])
    rows.append([f"Potok pelny {METRIC}"]
                + [tables.percent(recall.get(d)) for d in DATASETS])

    tables.show("Zbior testowy",
                ["Wielkosc"] + [LABEL[d] for d in DATASETS], rows,
                note="Myslnik w wierszu znacznika znaczy 'tego znacznika ten zbior nie "
                     "przypisuje', nie zero: VATEX nie niesie wymaga_osoby ani "
                     "wymaga_mimiki, bo jednozdaniowy opis klipu ich nie rozstrzyga. "
                     "Dwa ostatnie wiersze wypelniaja sie razem z przebiegami BAZY "
                     "i potoku pelnego.")


## 2. E2 - sygnał ruchu

BAZA wobec BAZY z sygnałem ruchu, kontrast konfirmacyjny na znaczniku `wymaga_ruchu`, rozbicie VATEX-a po przynależności klasy klipu do słownika Kinetics-400 oraz wkład krańcowy ruchu w potoku pełnym.

Rozbicie po K400 dotyczy wyłącznie VATEX-a: tylko klip niesie etykietę Kinetics, a poza tymi 400 nazwami dopasowana fraza czynnościowa nie ma na czym wylądować.

In [ ]:
E2C = "E2-C"
CONTRAST_E2 = "requirements:wymaga_ruchu"

if not missing([BASE, E2C]):
    group, columns = available([BASE, E2C])
    if group:
        header, rows, _ = variant_rows(group, columns, BASE, [BASE, E2C])
        rows += full_block("motion", columns)
        tables.show("E2: sygnal ruchu", header, rows)

        matching, rest, without = subsets(CONTRAST_E2, columns)
        if without:
            print(f"kontrast pominiety dla: {', '.join(LABEL[d] for d in without)}"
                  " - brak znacznika w tym zbiorze")
        if matching:
            contrast = compare.contrast_variant(on(group[E2C], matching),
                                                on(group[BASE], matching), METRIC,
                                                matching, rest)
            rows = []
            for dataset in matching:
                entry = contrast["per_dataset"][dataset]
                rows.append([LABEL[dataset], len(matching[dataset]),
                             tables.points(entry["matching_delta"]),
                             tables.points(entry["rest_delta"]),
                             tables.interval(entry)])
            rows.append(["polaczone odcinki",
                         sum(len(v) for d, v in matching.items()
                             if compare.inference_unit(d) == "episode"),
                         "-", "-", tables.interval(contrast["pooled"])])
            tables.show("E2: kontrast wymaga_ruchu",
                        ["Zbior", "Zapytan ze znacznikiem", "delta ze znacznikiem",
                         "delta bez znacznika", "Roznica [95% PU]"], rows,
                        note="Roznica roznic: efekt na zapytaniach ze znacznikiem minus "
                             "efekt na pozostalych. Potwierdzenie: "
                             f"{compare.confirmed(contrast)}.")

        if "vatex" in columns:
            rows = []
            for value, description in (("yes", "klasa w K400"), ("no", "klasa spoza K400")):
                inside, _ = compare.subset_ids("vatex", SPLIT, f"kinetics_vocab:{value}")
                rows.append([description, len(inside)]
                            + [tables.percent(compare.query_mean(
                                group[label]["vatex"]["per_query"], METRIC, inside))
                               for label in (BASE, E2C)])
            tables.show("E2: VATEX wedlug slownika Kinetics-400",
                        ["Podzbior", "Zapytan", f"{name(BASE)} {METRIC}",
                         f"{name(E2C)} {METRIC}"], rows,
                        note="Rozbicie opisowe, bez przedzialow: poza 400 nazwami "
                             "Kinetics dopasowana fraza czynnosciowa nie ma na czym "
                             "wyladowac, wiec sygnal ruchu jest tam z konstrukcji "
                             "nieaktywny.")

contribution_table("motion", "E2: wklad ruchu w potoku pelnym")


## 3. E3 - opisy scen

BAZA wobec BAZY z opisami BLIP i z opisami LLaVA, dwa kontrasty konfirmacyjne (po złożoności zapytania i po znaczniku `wymaga_scenerii`) oraz wkład krańcowy i efekt podmiany generatora opisów. Kontrast po złożoności jest ten sam, którym rozstrzygano E3 na zbiorze deweloperskim (`results/reports/e3_dev.json`).

In [ ]:
E3 = ["E3-B", "E3-C"]
CONTRASTS_E3 = [("complexity:Z", "zlozonosc Z"),
                ("requirements:wymaga_scenerii", "wymaga_scenerii")]

# Both variants have a test run. docs/05 (step 3) still does not read the
# REJECTED one as an individual contribution -- that role belongs to the swap
# effect -- so the block reports a gap but is not gated on it: the confirmatory
# contrast needs only the baseline and the winner.
missing(E3)
if not missing([BASE]):
    group, columns = available([BASE] + E3)
    if group:
        header, rows, _ = variant_rows(group, columns, BASE, [BASE] + E3)
        rows += full_block("caption", columns)
        tables.show("E3: generator opisow", header, rows)

        for spec, described in CONTRASTS_E3:
            matching, rest, without = subsets(spec, columns)
            if without:
                print(f"{described}: pominiete dla "
                      f"{', '.join(LABEL[d] for d in without)}")
            if not matching:
                continue
            rows = []
            for label in E3:
                if label not in group:
                    continue
                contrast = compare.contrast_variant(on(group[label], matching),
                                                    on(group[BASE], matching), METRIC,
                                                    matching, rest)
                for dataset in matching:
                    entry = contrast["per_dataset"][dataset]
                    rows.append([f"{label} / {LABEL[dataset]}",
                                 tables.points(entry["matching_delta"]),
                                 tables.points(entry["rest_delta"]),
                                 tables.interval(entry)])
                rows.append([f"{label} / polaczone", "-", "-",
                             tables.interval(contrast["pooled"])])
            tables.show(f"E3: kontrast {described}",
                        ["Konfiguracja", "delta w podzbiorze", "delta poza nim",
                         "Roznica [95% PU]"], rows)

contribution_table("caption", "E3: wklad opisow w potoku pelnym")


## 4. E4 - obiekty

BAZA wobec BAZY z YOLO11 i z YOLOE, wiersz E4-D, kontrast na znaczniku `wymaga_obiektu`, aktywacja sygnału oraz wkład krańcowy i efekt podmiany detektora.

**Wiersz E4-D czyta się inaczej niż pozostałe.** E4-D nie jest sygnałem, tylko etapem punktacji: przestawia pięćdziesiąt najlepszych fragmentów BAZY i reszty kolekcji nie rusza, więc jego sufit to Recall@50 BAZY i powyżej tego progu nie sięgnie żaden układ wag. Dlatego obok jego Recall@10 stoi Recall@50 BAZY, a nie różnica z przedziałem: to odległość do sufitu, nie efekt.

In [ ]:
E4 = ["E4-B", "E4-C"]
E4D = "E4-D"
CONTRAST_E4 = "requirements:wymaga_obiektu"
CEILING_K = 50            # E4-D re-ranks the fifty best of the base (query_detection.CANDIDATES)

# Both variants have a test run. docs/05 (step 3) still does not read the
# REJECTED one as an individual contribution -- that role belongs to the swap
# effect -- so the block reports a gap but is not gated on it: the confirmatory
# contrast needs only the baseline and the winner.
missing(E4)
if not missing([BASE]):
    group, columns = available([BASE] + E4)
    if group:
        header, rows, _ = variant_rows(group, columns, BASE, [BASE] + E4)
        rows += full_block("objects", columns)
        tables.show("E4: detektor obiektow", header, rows)

        matching, rest, without = subsets(CONTRAST_E4, columns)
        if without:
            print(f"kontrast pominiety dla: {', '.join(LABEL[d] for d in without)}")
        if matching:
            rows = []
            for label in E4:
                if label not in group:
                    continue
                contrast = compare.contrast_variant(on(group[label], matching),
                                                    on(group[BASE], matching), METRIC,
                                                    matching, rest)
                for dataset in matching:
                    entry = contrast["per_dataset"][dataset]
                    rows.append([f"{label} / {LABEL[dataset]}",
                                 tables.points(entry["matching_delta"]),
                                 tables.points(entry["rest_delta"]),
                                 tables.interval(entry)])
                rows.append([f"{label} / polaczone", "-", "-",
                             tables.interval(contrast["pooled"])])
            tables.show("E4: kontrast wymaga_obiektu",
                        ["Konfiguracja", "delta ze znacznikiem", "delta bez znacznika",
                         "Roznica [95% PU]"], rows)

        rows = []
        for label in E4:
            if label not in group:
                continue
            for dataset in columns:
                share = compare.activation_share(group[label][dataset])
                rows.append([f"{label} / {LABEL[dataset]}",
                             tables.percent(share.get("objects"))])
        if rows:
            tables.show("E4: zapytania z wlaczonym sygnalem obiektowym",
                        ["Konfiguracja", "Aktywacja [%]"], rows,
                        note="Udzial zapytan, dla ktorych sygnal mial co powiedziec: "
                             "fraza przedmiotowa zapytania dopasowala sie do nazwy "
                             "klasy powyzej progu. Zapytanie bez takiej frazy nie "
                             "wchodzi do wazenia wcale.")

# E4-D: its own row, read against the ceiling of the base rather than as a delta
if E4D in runs:
    group, columns = available([BASE, E4D])
    if len(group) == 2:
        rows = []
        for dataset in columns:
            base_queries = group[BASE][dataset]["per_query"]
            rows.append([LABEL[dataset],
                         tables.percent(compare.query_mean(base_queries, METRIC)),
                         tables.percent(metrics.recall_at(base_queries, CEILING_K)),
                         tables.percent(compare.query_mean(
                             group[E4D][dataset]["per_query"], METRIC))])
        tables.show(f"E4-D: detekcja sterowana zapytaniem ({E4D})",
                    ["Zbior", f"BAZA {METRIC}", f"BAZA recall@{CEILING_K}",
                     f"{E4D} {METRIC}"], rows,
                    note=f"E4-D przestawia {CEILING_K} najlepszych fragmentow BAZY i "
                         f"reszty kolekcji nie rusza, wiec recall@{CEILING_K} BAZY jest "
                         "jego sufitem. Bez przedzialu: to odleglosc do sufitu, nie "
                         "efekt sygnalu - E4-D nie ma wiersza w signals.npz ani wagi "
                         "w fuzji potoku.")
else:
    missing([E4D])

contribution_table("objects", "E4: wklad obiektow w potoku pelnym")


## 5. E5 - mechanizm mimiki, wyłącznie na serialach

BAZA wobec BAZY z regionami twarzy i z HSEmotion, kontrast na znaczniku `wymaga_mimiki`, wiersz opisowy na przecięciu zapytań aktywnych dla obu mechanizmów oraz wkład krańcowy i efekt podmiany.

**VATEX nie ma tu ani wiersza, ani kolumny, i to jest decyzja, nie brak danych.** `vatex_query_tags.csv` nie niesie kolumny `wymaga_mimiki`, bo jednozdaniowy opis klipu nie rozstrzyga wyrazu twarzy (`docs/02_dane_i_znaczniki.md`) - kolumna kontrastu tego wiersza byłaby pusta z konstrukcji, a mimiki dotyczy około 3% zapytań VATEX-a. Sygnał twarzy nie jest więc na VATEX-ie mierzony: `full_vatex.yaml` ma `face_regions: {enabled: false}`, a wariantów `full_no_face_regions_vatex` i `full_swap_face_regions_vatex` nie ma. Decyzja zapadła przed przebiegami testowymi, nie po nich.

In [ ]:
E5 = ["E5-B", "E5-C"]
CONTRAST_E5 = "requirements:wymaga_mimiki"
FACE_DATASETS = list(exp.datasets_of("face_regions"))

print("sygnal twarzy mierzony na: " + ", ".join(LABEL[d] for d in FACE_DATASETS))
print("poza pomiarem: " + ", ".join(LABEL[d] for d in DATASETS
                                    if d not in FACE_DATASETS)
      + " - decyzja rozdzialu 4, nie brak przebiegu")

# Both variants have a test run. docs/05 (step 3) still does not read the
# REJECTED one as an individual contribution -- that role belongs to the swap
# effect -- so the block reports a gap but is not gated on it: the confirmatory
# contrast needs only the baseline and the winner.
missing(E5)
if not missing([BASE]):
    group, columns = available([BASE] + E5)
    columns = [dataset for dataset in columns if dataset in FACE_DATASETS]
    if group and columns:
        header, rows, _ = variant_rows(group, columns, BASE, [BASE] + E5)
        rows += full_block("face_regions", columns)
        tables.show("E5: mechanizm mimiki", header, rows,
                    note="Tylko material serialowy: sygnal twarzy nie jest mierzony "
                         "na VATEX-ie.")

        matching, rest, without = subsets(CONTRAST_E5, columns)
        if without:
            print(f"kontrast pominiety dla: {', '.join(LABEL[d] for d in without)}")
        if matching:
            rows = []
            for label in E5:
                if label not in group:
                    continue
                contrast = compare.contrast_variant(on(group[label], matching),
                                                    on(group[BASE], matching), METRIC,
                                                    matching, rest)
                for dataset in matching:
                    entry = contrast["per_dataset"][dataset]
                    rows.append([f"{label} / {LABEL[dataset]}",
                                 tables.points(entry["matching_delta"]),
                                 tables.points(entry["rest_delta"]),
                                 tables.interval(entry)])
                rows.append([f"{label} / polaczone", "-", "-",
                             tables.interval(contrast["pooled"])])
            tables.show("E5: kontrast wymaga_mimiki",
                        ["Konfiguracja", "delta ze znacznikiem", "delta bez znacznika",
                         "Roznica [95% PU]"], rows)

        # the descriptive row: the queries BOTH mechanisms had something to say
        # about. Outside it the two are not answering the same question, so the
        # difference between their collection averages mixes two things. This is
        # the one block of the cell that genuinely needs both variants.
        if all(label in group for label in E5):
            rows = []
            for dataset in columns:
                both = (compare.active_ids(group["E5-B"][dataset], "face_regions")
                        & compare.active_ids(group["E5-C"][dataset], "face_regions"))
                rows.append([LABEL[dataset], len(both)]
                            + [tables.percent(compare.query_mean(
                                group[label][dataset]["per_query"], METRIC, both))
                               for label in E5])
            tables.show("E5: przeciecie zapytan aktywnych dla obu mechanizmow",
                        ["Zbior", "Zapytan", f"E5-B {METRIC}", f"E5-C {METRIC}"],
                        rows,
                        note="Wiersz opisowy, bez przedzialow: poza tym przecieciem "
                             "oba mechanizmy odpowiadaja na rozne zestawy zapytan, "
                             "wiec roznica ich srednich po calej kolekcji mieszalaby "
                             "jakosc mechanizmu z jego zasiegiem.")
        else:
            print("przeciecie pominiete: wymaga przebiegow obu mechanizmow, "
                  "a brakuje ktoregos z nich")

contribution_table("face_regions", "E5: wklad twarzy w potoku pelnym")


## 6. E6 - tożsamość

BAZA wobec BAZY z sygnałem tożsamości: na wszystkich zapytaniach i osobno na tych, które przywołują co najmniej jedną postać z profilem. Znacznik `wymaga_osoby` znaczy po decyzji autorki dokładnie "zapytanie przywołuje postać z profilem", więc podzbiór kontrastu i podzbiór `identities:>=1` są tym samym zbiorem i tabela pokazuje obie strony tej tożsamości zamiast jej ukrywać.

Podział po liczbie postaci jest w grupach rozłącznych: dokładnie jedna postać z profilem wobec dwóch lub więcej. Zagnieżdżone podzbiory (`identities:>=1` i `identities:>=2`) były wcześniejszym odczytem i wypadły, bo drugi zawierał się w pierwszym i wierszy nie dawało się zestawiać ze sobą.

Każda grupa niesie dwa wkłady: indywidualny (E6-B wobec BAZY) i krańcowy (potok pełny wobec potoku bez tożsamości). Sygnał sam na bazie i sygnał dokładany do potoku, w którym pozostałe już mówią, to dwie różne wielkości. Obie liczone jako różnica średnich po zapytaniach podzbioru, bez przedziału - to analiza eksploracyjna, nie kolejny test.

Delta jest więc różnicą dwóch pokazanych obok kolumn, a wiersze ważone liczbą zapytań sumują się do wiersza zapytań przywołujących postać z tabeli wyżej. Średnia różnic po odcinkach, którą liczą tabele z przedziałami, tej własności nie ma: każdy odcinek waży tyle samo niezależnie od tego, ile jego zapytań wpadło do grupy, więc wiersze nie sumowałyby się do wiersza, z którego zostały wycięte - na TBBT dawały -0,9 p.p. wobec +0,6 p.p. tamtego wiersza, z przeciwnym znakiem. W The Office grupa dwóch lub więcej postaci liczy w odcinku od jednego do dziesięciu zapytań, a odcinek z jednym zapytaniem wnosiłby do takiej średniej sto punktów. Kontrolę tego domknięcia blok wypisuje pod tabelą.

Blok wypisuje ponadto, na ilu odcinkach stoi kontrast `wymaga_osoby`: odcinek bez zapytań po jednej ze stron znacznika wypada z różnicy różnic, więc liczba ta bywa mniejsza niż liczba odcinków serialu.

Tożsamość jest mierzona wyłącznie na serialach - klipy VATEX-a nie mają powracających postaci z profilami.

In [ ]:
E6B = "E6-B"
CONTRAST_E6 = "requirements:wymaga_osoby"
IDENTITY_DATASETS = list(exp.datasets_of("identity"))


def on_queries(candidate, reference, chosen):
    """Difference of two subset means, both averaged over the QUERIES chosen.

    The unit of inference stays the episode wherever a number is INFERRED from:
    every interval in this notebook is still the t interval over episodes. This
    is the descriptive counterpart, and it is what the split by character count
    needs -- a mean over episodes is not additive over subsets of the queries,
    so rows cut out of one row would not add back up to it.
    """
    after = compare.query_mean(candidate["per_query"], METRIC, chosen)
    before = compare.query_mean(reference["per_query"], METRIC, chosen)
    return None if after is None or before is None else after - before

print("sygnal tozsamosci mierzony na: " + ", ".join(LABEL[d] for d in IDENTITY_DATASETS))

if not missing([BASE, E6B]):
    group, columns = available([BASE, E6B])
    columns = [dataset for dataset in columns if dataset in IDENTITY_DATASETS]
    if group and columns:
        header, rows, _ = variant_rows(group, columns, BASE, [BASE, E6B])
        rows += full_block("identity", columns)
        tables.show("E6: sygnal tozsamosci", header, rows)

        with_person, _, without = subsets("identities:>=1", columns)
        if without:
            print(f"brak zapytan z postacia w: {', '.join(LABEL[d] for d in without)}")
        if with_person:
            result = compare.compare_variant(on(group[E6B], with_person),
                                             on(group[BASE], with_person), METRIC,
                                             with_person)
            rows = [[LABEL[dataset], len(with_person[dataset]),
                     tables.percent(result["per_dataset"][dataset]["reference_value"]),
                     tables.percent(result["per_dataset"][dataset]["value"]),
                     tables.interval(result["per_dataset"][dataset])]
                    for dataset in with_person]
            rows.append(["polaczone odcinki",
                         sum(len(v) for v in with_person.values()), "-", "-",
                         tables.interval(result["pooled"])])
            tables.show("E6: zapytania przywolujace postac",
                        ["Zbior", "Zapytan", f"BAZA {METRIC}", f"{E6B} {METRIC}",
                         "delta [95% PU]"], rows)

        matching, rest, empty = subsets(CONTRAST_E6, columns)
        if matching:
            contrast = compare.contrast_variant(on(group[E6B], matching),
                                                on(group[BASE], matching), METRIC,
                                                matching, rest)
            rows = [[LABEL[dataset], len(matching[dataset]),
                     tables.points(contrast["per_dataset"][dataset]["matching_delta"]),
                     tables.points(contrast["per_dataset"][dataset]["rest_delta"]),
                     tables.interval(contrast["per_dataset"][dataset])]
                    for dataset in matching]
            rows.append(["polaczone odcinki", sum(len(v) for v in matching.values()),
                         "-", "-", tables.interval(contrast["pooled"])])
            tables.show("E6: kontrast wymaga_osoby",
                        ["Zbior", "Zapytan ze znacznikiem", "delta ze znacznikiem",
                         "delta bez znacznika", "Roznica [95% PU]"], rows,
                        note="Po decyzji autorki wymaga_osoby znaczy 'zapytanie "
                             "przywoluje postac z profilem', wiec ten podzbior i "
                             "identities:>=1 to ten sam zbior zapytan. "
                             f"Potwierdzenie: {compare.confirmed(contrast)}.")

            # How many EPISODES the contrast actually rests on. An episode with
            # no tagged query, or none outside the tag, has no difference on one
            # of the two sides and drops out of the difference of differences --
            # so this n is not the episode count of the series, and the two
            # numbers belong next to each other.
            print("odcinkow w kontrascie wymaga_osoby: " + ", ".join(
                f"{LABEL[dataset]} {contrast['per_dataset'][dataset]['n']} z "
                f"{len(compare.episode_means(group[BASE][dataset]['per_query'], METRIC))}"
                for dataset in matching))

        # The split by how many characters a query names, in DISJOINT groups: a
        # query naming two of them belongs to one row and not to both. Nested
        # subsets (>=1 and >=2) were the earlier reading and their rows could not
        # be held against each other, the first containing the second.
        #
        # Two contributions per row, because the signal is worth two different
        # things: ALONE on the base, and AT THE MARGIN of the full pipeline where
        # the other signals already speak. Both come without an interval: this is
        # exploratory, not one more test.
        #
        # Both are differences of the two subset means over QUERIES, and NOT the
        # mean of per-episode differences the tables above report. The rows here
        # are subsets of one row there, and an episode mean is not additive over
        # such subsets: every episode weighs the same however few of its queries
        # fall in the group, so the rows would not add up to the row they were
        # cut out of. On TBBT they did not -- weighted by their query counts they
        # came to -0,9 p.p. against the +0,6 p.p. of the row itself, the opposite
        # SIGN. Over queries they add up exactly and the delta is the difference
        # of the two columns printed beside it. The Office group "two or more" is
        # why this matters: it holds between one and ten queries per episode, and
        # an episode carrying one of them moves an equally weighted mean by a
        # hundred points.
        one_plus, _, _ = subsets("identities:>=1", columns)
        two_plus, _, _ = subsets("identities:>=2", columns)
        marginal, _ = ({}, []) if missing([FULL, exp.no_label("identity")]) else \
            available([FULL, exp.no_label("identity")])
        rows = []
        for dataset in columns:
            if dataset not in one_plus:
                continue
            several = two_plus.get(dataset, set())
            for described, chosen in (("dokladnie jedna postac",
                                       one_plus[dataset] - several),
                                      ("dwie lub wiecej postaci", several)):
                if not chosen:
                    continue
                added = on_queries(group[E6B][dataset], group[BASE][dataset], chosen)
                at_margin = None
                if all(dataset in marginal.get(label, {})
                       for label in (FULL, exp.no_label("identity"))):
                    at_margin = on_queries(marginal[FULL][dataset],
                                           marginal[exp.no_label("identity")][dataset],
                                           chosen)
                rows.append([described, LABEL[dataset], len(chosen),
                             tables.percent(compare.query_mean(
                                 group[BASE][dataset]["per_query"], METRIC, chosen)),
                             tables.percent(compare.query_mean(
                                 group[E6B][dataset]["per_query"], METRIC, chosen)),
                             tables.points(added), tables.points(at_margin)])
        if rows:
            tables.show("E6: podzial po liczbie postaci",
                        ["Podzbior", "Zbior", "Zapytan", f"BAZA {METRIC}",
                         f"{E6B} {METRIC}", "delta ind. [p.p.]",
                         "delta krancowa [p.p.]"], rows, align="llrrrrr",
                        note="Grupy sa ROZLACZNE: 'dokladnie jedna postac' to "
                             "identities:>=1 bez identities:>=2, wiec zapytanie stoi "
                             "w jednym wierszu, nie w dwoch. Wszystkie cztery liczby "
                             "wiersza sa srednimi po ZAPYTANIACH podzbioru, wiec "
                             "delta jest roznica dwoch pokazanych obok kolumn, a "
                             "wiersze wazone liczba zapytan sumuja sie do wiersza "
                             "zapytan przywolujacych postac. 'delta ind.' to E6-B "
                             "wobec BAZY, 'delta krancowa' - potok pelny wobec "
                             "potoku bez tozsamosci; bez przedzialow, bo to analiza "
                             "eksploracyjna, nie test. Tabele z przedzialami licza "
                             "delte jako srednia roznic po odcinkach i roznia sie od "
                             "tych o dziesiate czesci punktu. Myslnik w ostatniej "
                             "kolumnie znaczy, ze brakuje przebiegu potoku pelnego "
                             "albo full_no_identity.")

            # The closure the note above claims, checked rather than promised:
            # the rows weighted by their query counts have to give the delta of
            # the row they were cut out of. It is the property the per-episode
            # mean did not have, so it is worth printing every time.
            for dataset in columns:
                if dataset not in one_plus:
                    continue
                several = two_plus.get(dataset, set())
                parts = [part for part in (one_plus[dataset] - several, several) if part]
                # A part whose queries no run covers leaves the check undecided,
                # so it prints a dash instead of a sum quietly short of one row.
                weighted = [(len(part), on_queries(group[E6B][dataset],
                                                   group[BASE][dataset], part))
                            for part in parts]
                known = [(size, delta) for size, delta in weighted if delta is not None]
                summed = (sum(size * delta for size, delta in known)
                          / len(one_plus[dataset])
                          if len(known) == len(weighted) else None)
                whole = on_queries(group[E6B][dataset], group[BASE][dataset],
                                   one_plus[dataset])
                print(f"domkniecie {LABEL[dataset]}: wiersze wazone liczba zapytan "
                      f"{tables.points(summed)} p.p., wiersz zapytan przywolujacych "
                      f"postac {tables.points(whole)} p.p.")

contribution_table("identity", "E6: wklad tozsamosci w potoku pelnym")


### 6a. Wkład krańcowy tożsamości na zapytaniach przywołujących postać

Uściślenie wiersza wkładu krańcowego z bloku wyżej. Tamten liczy się po całej kolekcji, a sygnał tożsamości odzywa się wyłącznie do zapytań przywołujących postać z profilem; poza nimi potok pełny i potok bez tożsamości są tym samym potokiem, więc średnia po całej kolekcji rozcieńcza efekt zapytaniami, których sygnał w ogóle nie dotknął. Pierwsza tabela liczy tę samą różnicę na podzbiorze `identities:>=1`, osobno na serial i na połączonych odcinkach, z przedziałem t po odcinkach. Kolumna "odcinków" jest $n$ tego przedziału.

Druga tabela sprawdza założenie, na którym pierwsza stoi: na zapytaniach bez postaci z profilem oba przebiegi mają dawać identyczne rankingi. Nie wynika to z samego kodu punktacji, bo wyłączenie sygnału przelicza wagi pozostałych, więc sprawdza się to na wynikach. Niezerowe przejście $0\to1$ albo $1\to0$ znaczy, że założenie jest fałszywe, a nie że przebieg jest zły.

In [ ]:
NO_IDENTITY = exp.no_label("identity")
IDENTITY_DATASETS = list(exp.datasets_of("identity"))

if not missing([FULL, NO_IDENTITY]):
    marginal, marginal_columns = available([FULL, NO_IDENTITY])
    marginal_columns = [dataset for dataset in marginal_columns
                        if dataset in IDENTITY_DATASETS]
    named, unnamed, absent = subsets("identities:>=1", marginal_columns)
    if absent:
        print(f"brak zapytan z postacia w: {', '.join(LABEL[d] for d in absent)}")

    if named:
        # The marginal contribution of the block above, narrowed to the queries the
        # signal is active for: same pair, same arithmetic, fewer queries. The
        # interval is still the t interval over episodes -- over the episodes
        # that carry such a query, which is what the "Odcinkow" column says.
        result = compare.compare_variant(on(marginal[FULL], named),
                                         on(marginal[NO_IDENTITY], named),
                                         METRIC, named)
        rows = [[LABEL[dataset], len(named[dataset]),
                 result["per_dataset"][dataset]["n"],
                 tables.percent(result["per_dataset"][dataset]["value"]),
                 tables.percent(result["per_dataset"][dataset]["reference_value"]),
                 tables.interval(result["per_dataset"][dataset])]
                for dataset in named]
        rows.append(["polaczone odcinki", sum(len(v) for v in named.values()),
                     result["pooled"]["n"], "-", "-",
                     tables.interval(result["pooled"])])
        tables.show("E6: wklad krancowy tozsamosci na zapytaniach z postacia",
                    ["Zbior", "Zapytan", "Odcinkow", f"potok PELNY {METRIC}",
                     f"{name(NO_IDENTITY)} {METRIC}", "delta [95% PU]"],
                    rows, align="lrrrrl",
                    note="Potok PELNY wobec potoku bez tozsamosci, na zapytaniach "
                         "identities:>=1. Poziomy R@10 to srednie po zapytaniach tego "
                         "podzbioru, delta - srednia roznic po odcinkach z przedzialem "
                         "t. 'Odcinkow' jest n przedzialu: odcinek bez takiego "
                         "zapytania do niego nie wchodzi.")

        # What the table above rests on: outside the subset the identity signal
        # is inactive, so the two pipelines have to score those queries
        # identically. Removing a signal recomputes the weights of the others, so
        # this does not follow from the scoring code and is checked on results.
        rows = []
        for dataset in marginal_columns:
            outside = unnamed.get(dataset)
            if not outside:
                continue
            counts = compare.transitions(marginal[FULL][dataset],
                                         marginal[NO_IDENTITY][dataset], outside)
            ranking = {int(r["desc_id"]): [hit["fragment"] for hit in r["top"]]
                       for r in marginal[FULL][dataset]["per_query"]}
            reference = {int(r["desc_id"]): [hit["fragment"] for hit in r["top"]]
                         for r in marginal[NO_IDENTITY][dataset]["per_query"]}
            same = sum(ranking[desc_id] == reference[desc_id] for desc_id in outside)
            rows.append([LABEL[dataset], len(outside), counts["0->1"], counts["1->0"],
                         f"{same}/{len(outside)}",
                         "zgodne" if not (counts["0->1"] or counts["1->0"])
                         and same == len(outside)
                         else "NIEZGODNE - popraw rozdzial 6"])
        if rows:
            tables.show("E6: zapytania bez postaci z profilem - kontrola",
                        ["Zbior", "Zapytan", "0->1", "1->0", "Identyczny ranking",
                         "Werdykt"], rows, align="l" + "r" * 4 + "l",
                        note="Potok PELNY wobec potoku bez tozsamosci na zapytaniach, "
                             "ktorych sygnal tozsamosci nie dotyka. 'Identyczny "
                             "ranking' liczy zapytania, dla ktorych cala zapisana "
                             "lista fragmentow jest ta sama w obu przebiegach, nie "
                             "tylko trafienie w pierwszej dziesiatce.")
